In [1]:
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.inspection import partial_dependence
import numpy as np
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.metrics import (
    silhouette_score, 
    r2_score,
)

import xgboost as xgb
from umap import UMAP
import shap

# =========================
# Custom utilities
# =========================
import utils.cross_validation as cval
import utils.umap_utils as uutil
import utils.exp_utils as exp

# =========================
# Optional helpers (math functions already included above)
# =========================
from math import radians, sin, cos, sqrt, asin
import math

/home/qli/Projects/env1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
"""
Double/Debiased Machine Learning: effect of functional diversity (Raos_Q)
on transformed NPP, using FIA-derived data (fd_df), via the DoubleML package.

Install first if needed:
    pip install DoubleML xgboost --break-system-packages
"""

import numpy as np
import pandas as pd
import xgboost as xgb
import doubleml as dml
from doubleml import DoubleMLData, DoubleMLPLR



In [2]:
# Essential Params and Dicts

color_map = {
    'Species Richness': 'grey',
    'Shannon Diversity': "#8040a0",
    "Raos_Q": 'darkolivegreen',
    "Simpson's Index":  "#cca9dd",
    'Functional_Evenness': "#94BC92"
}
biome_colors = {
    'Temperate broadleaf forests': "#4881BE",   # Sea Green (mid-tone green)
    'Temperate conifer forests':   "#016742",   # Very Dark Green (near-black green)
    'Temperate grasslands':        '#DAA520',   # Goldenrod (yellow-brown)
    'Xeric shrublands':             '#B5651D',   # Burnt Orange-Brown (desert)
    'Boreal and Tundra forests':    '#4169E1',   # Royal Blue (cool, distinct from green)
    'Mediterranean woodlands':      '#9ACD32',   # Yellow-Green (olive, warm-leaning)
    'Tropical':                     '#FF00FF',   # Magenta (maximally distinct, non-natural but unmistakable)
    'NaN':                          '#D3D3D3'    # Light Gray
}

with open("config/ex_config.yaml", "r") as f:
    config = yaml.safe_load(f)

biome_mapping = config["biome_mapping"]
random_key=config["random_key"]

biome_list=["Temperate broadleaf forests", "Temperate conifer forests", "Temperate grasslands",
            "Xeric shrublands", "Boreal and Tundra forests", "Tropical",
            "Mediterranean woodlands"]

In [6]:
fd_df = pd.read_csv('data/final/final_dataset.csv')
PID_df= pd.read_csv('data/lookup/PID_location_all.csv')

fd_df=fd_df.merge(PID_df[['PID','lat','lon', 'biome']], on='PID', how='left')

fd_df.dropna(subset=['transformed npp', 'Raos_Q', 'Functional_Evenness', 'Soil Moisture',
                    'Species Richness', 'Shannon Diversity', "Simpson's Index", "pet_std"], inplace=True)
fd_df = fd_df[fd_df["treecover2000"] > 60]


fd_df.drop_duplicates(subset=['PID'], inplace=True)

npp_df=pd.read_csv('data/processed/PID_npp_volatility.csv')

npp_df=npp_df[['PID', 'mean']]

fd_df=fd_df.merge(npp_df, on='PID', how='left')

ecoregions=cval.process_ecoregion("data/Ecoregions/Ecoregions2017.shp")

ecoregions=ecoregions[['ECO_NAME', 'geometry']]
#Preprocessing the data for Random forest regression

fd_df['biome'] = fd_df['biome'].map(biome_mapping) # Only run this once ! 

biome_dfs = {k: v for k, v in fd_df.groupby('biome')}

fd_df = fd_df[fd_df["WSCI"] != 0]


In [33]:
fd_df.columns

Index(['PID', 'Raos_Q', 'Functional_Evenness', 'Functional_Richness',
       'Species Richness', 'Shannon Diversity', 'Simpson's Index', 'WSCI',
       'Annual Temp', 'Annual Precipitation', 'Precipitation Seasonality',
       'Soil Moisture', 'SOC Content', 'Burnt Areas Probability', 'Elevation',
       'Slope', 'Depth to Bedrock', 'Northness', 'transformed npp',
       'treecover2000', 'pet_std', 'disturbance_value', 'land_cover_value',
       'lat', 'lon', 'biome', 'mean'],
      dtype='str')

In [40]:

# ---------------------------------------------------------------------------
# 2. Define outcome, treatment, and confounders
# ---------------------------------------------------------------------------
y_col = 'transformed npp'
d_col = "Simpson's Index"

# Columns excluded from the confounder set X:
#   - the outcome and treatment themselves
#   - identifiers / non-causal metadata (PID, lat, lon)
#   - other diversity metrics you don't want acting as controls
#     (Functional_Richness, Functional_Evenness) -- drop these from the
#     exclusion list instead if you DO want them treated as confounders
cols_to_exclude = [
    y_col, d_col, 'Functional_Richness', 'Functional_Evenness', 'Raos_Q', 'Shannon Diversity', 'Species Richness',
    'PID', 'lat', 'lon', 'biome', 'treecover2000'
]

x_cols = [c for c in fd_df.columns if c not in cols_to_exclude]
print("Confounder columns (X):", x_cols)

Confounder columns (X): ['WSCI', 'Annual Temp', 'Annual Precipitation', 'Precipitation Seasonality', 'Soil Moisture', 'SOC Content', 'Burnt Areas Probability', 'Elevation', 'Slope', 'Depth to Bedrock', 'Northness', 'pet_std', 'disturbance_value', 'land_cover_value', 'mean']


In [41]:
# ---------------------------------------------------------------------------
# 3. Build the DoubleMLData object
# ---------------------------------------------------------------------------
dml_data = DoubleMLData(
    fd_df,
    y_col=y_col,
    d_cols=d_col,
    x_cols=x_cols
)
print(dml_data)


================== DoubleMLData Object ==================

------------------ Data summary      ------------------
Outcome variable: transformed npp
Treatment variable(s): ["Simpson's Index"]
Covariates: ['WSCI', 'Annual Temp', 'Annual Precipitation', 'Precipitation Seasonality', 'Soil Moisture', 'SOC Content', 'Burnt Areas Probability', 'Elevation', 'Slope', 'Depth to Bedrock', 'Northness', 'pet_std', 'disturbance_value', 'land_cover_value', 'mean']
Instrument variable(s): None
No. Observations: 46117
------------------ DataFrame info    ------------------
<class 'pandas.DataFrame'>
Index: 46117 entries, 0 to 47367
Columns: 27 entries, PID to mean
dtypes: float64(21), int64(4), str(2)
memory usage: 11.6 MB



In [42]:

# ---------------------------------------------------------------------------
# 4. Specify nuisance learners (ML models for Y~X and D~X)
# ---------------------------------------------------------------------------
xgb_params = dict(
    objective='reg:squarederror',
    learning_rate=0.01,
    max_depth=6,
    min_child_weight=2,
    gamma=0.1,
    reg_lambda=1.0,
    reg_alpha=0.0,
    tree_method='hist',   # you listed 'hist' then 'approx' — a dict can only keep one, pick one
    n_estimators=500,
    subsample=0.7,
    random_state=42,
)

ml_l = xgb.XGBRegressor(**xgb_params)   # nuisance model for Y ~ X  (outcome regression)
ml_m = xgb.XGBRegressor(**xgb_params)   # nuisance model for D ~ X  (treatment regression)

In [43]:
# ---------------------------------------------------------------------------
# 5. Fit the Partially Linear Regression (PLR) DML model
# ---------------------------------------------------------------------------
np.random.seed(42)

plr_model = DoubleMLPLR(
    dml_data,
    ml_l=ml_l,
    ml_m=ml_m,
    n_folds=5,          # cross-fitting folds
    n_rep=10,           # repeat cross-fitting 10x and average -> more stable estimate
    score='partialling out'
)

plr_model.fit()

In [44]:
# ---------------------------------------------------------------------------
# 6. Results
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("DML RESULTS: Effect of Raos_Q (functional diversity) on transformed NPP")
print("=" * 60)
print(plr_model.summary)

theta = plr_model.coef[0]
se = plr_model.se[0]
ci = plr_model.confint()
print(f"\ntheta (effect size): {theta:.4f}")
print(f"Std. Error:          {se:.4f}")
print(f"95% CI:               [{ci.iloc[0,0]:.4f}, {ci.iloc[0,1]:.4f}]")
print(f"p-value:              {plr_model.pval[0]:.4g}")



DML RESULTS: Effect of Raos_Q (functional diversity) on transformed NPP
                     coef   std err         t    P>|t|     2.5 %    97.5 %
Simpson's Index -0.009504  0.006984 -1.360821  0.17357 -0.023191  0.004184

theta (effect size): -0.0095
Std. Error:          0.0070
95% CI:               [-0.0232, 0.0042]
p-value:              0.1736


In [45]:



# # ---------------------------------------------------------------------------
# # 7. Robustness check: swap in a different nuisance learner (Random Forest)
# # ---------------------------------------------------------------------------
# from sklearn.ensemble import RandomForestRegressor

# rf_params = dict(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
# ml_l_rf = RandomForestRegressor(**rf_params)
# ml_m_rf = RandomForestRegressor(**rf_params)

# plr_model_rf = DoubleMLPLR(
#     dml_data,
#     ml_l=ml_l_rf,
#     ml_m=ml_m_rf,
#     n_folds=5,
#     n_rep=10,
#     score='partialling out'
# )
# plr_model_rf.fit()

# print("\n" + "=" * 60)
# print("ROBUSTNESS CHECK: Random Forest nuisance learners")
# print("=" * 60)
# print(f"theta (XGBoost):       {theta:.4f}  (SE={se:.4f})")
# print(f"theta (Random Forest): {plr_model_rf.coef[0]:.4f}  (SE={plr_model_rf.se[0]:.4f})")
# print("-> If these are close, your estimate is robust to nuisance-model choice.")

# # ---------------------------------------------------------------------------
# # 8. Optional: check nuisance model out-of-sample fit quality
# # ---------------------------------------------------------------------------
# # Poor nuisance fits undermine DML's guarantees. Inspect predictive performance:
# print("\nNuisance model prediction summary (XGBoost run):")
# print(plr_model.evaluate_learners())